# 5bii: Node-Level and Structural Graph Features vs Prediction Quality

Extends 5b's aggregate connectivity analysis to **per-stock** and **structural** graph features.

**Key advantage:** Cross-sectional analysis (88 stocks × 1400 windows) provides much more
statistical power than the time-series analysis in 5b (1400 windows only).

**Analyses:**
1. Degree vs per-stock prediction quality
2. Hub stock analysis
3. Clustering coefficient vs prediction
4. Node centrality vs prediction
5. Modularity over time
6. Spectral gap over time

## 1. Setup

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import networkx as nx
from settings.default import ALL_TICKERS, BBG_SECTORS
print("Imports ready")

## 2. Load Data

In [ ]:
from gml.experiment_utils import load_experiment_results

# Load GCN Rolling results
GCN_EXPERIMENT = "3_GCN_rolling_pearson"
GCN_CONFIG = "lb20_th0.4_equity"  # Adjust to match your config
GCN_SEED = 40

print(f"Loading {GCN_EXPERIMENT}/{GCN_CONFIG}/seed_{GCN_SEED}")
gcn_results = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)

gcn_adj = gcn_results["adjacency"]         # (num_windows, 88, 88)
gcn_returns_df = gcn_results["captured_returns"]  # DataFrame with per-stock returns
gcn_dates = gcn_results["test_dates"]
gcn_daily = gcn_results["daily_returns"]

tickers = sorted(ALL_TICKERS)
sectors = [BBG_SECTORS.get(t, "Unknown") for t in tickers]

print(f"Adjacency: {gcn_adj.shape}")
print(f"Test dates: {len(gcn_dates)}")
print(f"Tickers: {len(tickers)}")

## 3. Compute Per-Window Node Features

In [ ]:
def compute_node_features(adjacencies):
    """Compute per-node graph features for each window."""
    n_windows = len(adjacencies)
    n_nodes = adjacencies[0].shape[0]
    
    degrees = np.zeros((n_windows, n_nodes))
    clustering = np.zeros((n_windows, n_nodes))
    eigenvector_cent = np.zeros((n_windows, n_nodes))
    
    for t in range(n_windows):
        adj = adjacencies[t].copy()
        np.fill_diagonal(adj, 0)
        
        # Degree (number of connections)
        degrees[t] = (adj > 0).sum(axis=1)
        
        # Build networkx graph for clustering and centrality
        G = nx.from_numpy_array(adj)
        
        # Clustering coefficient
        cc = nx.clustering(G)
        clustering[t] = [cc[i] for i in range(n_nodes)]
        
        # Eigenvector centrality
        try:
            ec = nx.eigenvector_centrality_numpy(G)
            eigenvector_cent[t] = [ec[i] for i in range(n_nodes)]
        except:
            eigenvector_cent[t] = np.nan
    
    if t % 200 == 0:
        print(f"  Processed {t+1}/{n_windows} windows")
    
    return {
        "degree": degrees,
        "clustering": clustering,
        "eigenvector_centrality": eigenvector_cent,
    }

print("Computing node features for all windows (this may take a few minutes)...")
node_features = compute_node_features(gcn_adj)
print(f"Done. Shapes: degree={node_features['degree'].shape}")

## 4. Per-Stock Captured Returns

In [ ]:
# Build per-stock, per-window captured returns matrix
# captured_returns_sw has columns: time, identifier, position, returns, captured_returns
cr = gcn_returns_df.copy()
cr["time"] = pd.to_datetime(cr["time"])

# Pivot to (dates, tickers) matrix
per_stock_returns = cr.pivot_table(
    index="time", columns="identifier", values="captured_returns", aggfunc="first"
)

# Align with test window dates
window_dates = pd.to_datetime(gcn_dates)
per_stock_returns = per_stock_returns.reindex(window_dates)

# Ensure ticker order matches
available_tickers = [t for t in tickers if t in per_stock_returns.columns]
per_stock_returns = per_stock_returns[available_tickers]

print(f"Per-stock returns matrix: {per_stock_returns.shape}")
print(f"Tickers matched: {len(available_tickers)} / {len(tickers)}")

## 5. Analysis 1: Degree vs Per-Stock Prediction Quality

In [ ]:
# For each window: correlate degree with captured return across 88 stocks
per_window_corrs = []
for t in range(len(gcn_adj)):
    deg = node_features["degree"][t]
    ret = per_stock_returns.iloc[t].values if t < len(per_stock_returns) else None
    
    if ret is not None and not np.all(np.isnan(ret)):
        valid = ~np.isnan(ret)
        if valid.sum() > 10:
            r, p = stats.pearsonr(deg[valid], ret[valid])
            per_window_corrs.append({"date": window_dates[t], "r": r, "p": p})

corr_df = pd.DataFrame(per_window_corrs)
corr_df = corr_df.set_index("date")

print(f"Per-window degree-return correlation:")
print(f"  Mean r: {corr_df['r'].mean():.4f}")
print(f"  Std r:  {corr_df['r'].std():.4f}")
print(f"  % windows with p<0.05: {(corr_df['p'] < 0.05).mean():.1%}")
print(f"  % windows with r>0: {(corr_df['r'] > 0).mean():.1%}")

# t-test: is mean correlation significantly different from zero?
t_stat, p_val = stats.ttest_1samp(corr_df["r"].dropna(), 0)
print(f"
  t-test (mean r != 0): t={t_stat:.3f}, p={p_val:.4f}")
print(f"  {'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'}")

In [ ]:
# Plot: rolling average of degree-return correlation over time
fig, ax = plt.subplots(figsize=(16, 5))
rolling_r = corr_df["r"].rolling(60).mean()
ax.plot(rolling_r.index, rolling_r.values, lw=1)
ax.axhline(y=0, color="black", ls="--", lw=0.5)
ax.fill_between(rolling_r.index, rolling_r.values, 0,
                where=rolling_r > 0, color="green", alpha=0.2)
ax.fill_between(rolling_r.index, rolling_r.values, 0,
                where=rolling_r <= 0, color="red", alpha=0.2)
ax.set_ylabel("Rolling 60-Day Mean Correlation (degree vs return)")
ax.set_title("Does Higher Degree Predict Better Returns? (per-window cross-sectional correlation)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Analysis 2: Hub Stock Analysis

In [ ]:
# Identify hub stocks (persistently high degree)
avg_degree = node_features["degree"].mean(axis=0)  # average degree per stock across all windows
degree_threshold = np.percentile(avg_degree, 75)  # top quartile = hubs

hub_mask = avg_degree >= degree_threshold
hub_tickers = [t for t, is_hub in zip(available_tickers, hub_mask) if is_hub]
non_hub_tickers = [t for t, is_hub in zip(available_tickers, hub_mask) if not is_hub]

print(f"Hub threshold (75th percentile): {degree_threshold:.1f} average connections")
print(f"Hub stocks ({len(hub_tickers)}): {', '.join(hub_tickers)}")
print(f"
Hub stocks and sectors:")
for t in hub_tickers:
    print(f"  {t}: {BBG_SECTORS.get(t, 'Unknown')} (avg degree: {avg_degree[available_tickers.index(t)]:.1f})")

In [ ]:
# Compare hub vs non-hub average captured returns
hub_returns = per_stock_returns[hub_tickers].mean(axis=1)
non_hub_returns = per_stock_returns[non_hub_tickers].mean(axis=1)

from empyrical import sharpe_ratio

print(f"
Performance comparison:")
print(f"{'Metric':<25} {'Hub Stocks':>15} {'Non-Hub Stocks':>15}")
print("-" * 55)
print(f"{'Mean Daily Return':<25} {hub_returns.mean()*100:>15.4f}% {non_hub_returns.mean()*100:>15.4f}%")
print(f"{'Daily Volatility':<25} {hub_returns.std()*100:>15.4f}% {non_hub_returns.std()*100:>15.4f}%")
print(f"{'Sharpe Ratio':<25} {sharpe_ratio(hub_returns):>15.3f} {sharpe_ratio(non_hub_returns):>15.3f}")

# Significance test
t_stat, p_val = stats.ttest_ind(hub_returns.dropna(), non_hub_returns.dropna())
print(f"
t-test (hub vs non-hub returns): t={t_stat:.3f}, p={p_val:.4f}")
print(f"{'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'}")

In [ ]:
# Scatter: average degree vs average captured return per stock
avg_returns = per_stock_returns.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 7))
for i, t in enumerate(available_tickers):
    sector = BBG_SECTORS.get(t, "Unknown")
    color = {"Information Technology": "#1f77b4", "Healthcare": "#2ca02c",
             "Financials": "#ff7f0e", "Consumer Discretionary": "#d62728",
             "Consumer Staples": "#9467bd", "Industrials": "#8c564b",
             "Communication Services": "#e377c2", "Energy": "#7f7f7f"}.get(sector, "#333333")
    ax.scatter(avg_degree[i], avg_returns.iloc[i] * 100, color=color, s=40, alpha=0.7)
    ax.annotate(t, (avg_degree[i], avg_returns.iloc[i] * 100), fontsize=6, alpha=0.7)

# Regression line
z = np.polyfit(avg_degree, avg_returns.values * 100, 1)
p = np.poly1d(z)
x_line = np.linspace(avg_degree.min(), avg_degree.max(), 100)
ax.plot(x_line, p(x_line), "r--", lw=2)

r, pval = stats.pearsonr(avg_degree, avg_returns.values)
ax.set_xlabel("Average Degree (connections)")
ax.set_ylabel("Average Daily Captured Return (%)")
ax.set_title(f"Degree vs Prediction Quality (r={r:.3f}, p={pval:.3f})")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Analysis 3: Clustering Coefficient vs Prediction

In [ ]:
# Per-window correlation: clustering coefficient vs captured return
cluster_corrs = []
for t in range(len(gcn_adj)):
    cc = node_features["clustering"][t]
    ret = per_stock_returns.iloc[t].values if t < len(per_stock_returns) else None
    
    if ret is not None and not np.all(np.isnan(ret)):
        valid = ~np.isnan(ret) & ~np.isnan(cc)
        if valid.sum() > 10:
            r, p = stats.pearsonr(cc[valid], ret[valid])
            cluster_corrs.append({"r": r, "p": p})

cluster_corr_df = pd.DataFrame(cluster_corrs)

print(f"Per-window clustering-return correlation:")
print(f"  Mean r: {cluster_corr_df['r'].mean():.4f}")
print(f"  % windows with p<0.05: {(cluster_corr_df['p'] < 0.05).mean():.1%}")

t_stat, p_val = stats.ttest_1samp(cluster_corr_df["r"].dropna(), 0)
print(f"  t-test (mean r != 0): t={t_stat:.3f}, p={p_val:.4f}")
print(f"  {'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'}")
print(f"  Interpretation: {'stocks in tighter clusters get better predictions' if cluster_corr_df['r'].mean() > 0 else 'stocks in tighter clusters get WORSE predictions'}")

## 8. Analysis 4: Eigenvector Centrality vs Prediction

In [ ]:
# Per-window correlation: eigenvector centrality vs captured return
cent_corrs = []
for t in range(len(gcn_adj)):
    ec = node_features["eigenvector_centrality"][t]
    ret = per_stock_returns.iloc[t].values if t < len(per_stock_returns) else None
    
    if ret is not None and not np.all(np.isnan(ret)) and not np.all(np.isnan(ec)):
        valid = ~np.isnan(ret) & ~np.isnan(ec)
        if valid.sum() > 10:
            r, p = stats.pearsonr(ec[valid], ret[valid])
            cent_corrs.append({"r": r, "p": p})

cent_corr_df = pd.DataFrame(cent_corrs)

print(f"Per-window centrality-return correlation:")
print(f"  Mean r: {cent_corr_df['r'].mean():.4f}")
print(f"  % windows with p<0.05: {(cent_corr_df['p'] < 0.05).mean():.1%}")

t_stat, p_val = stats.ttest_1samp(cent_corr_df["r"].dropna(), 0)
print(f"  t-test (mean r != 0): t={t_stat:.3f}, p={p_val:.4f}")
print(f"  {'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'}")

## 9. Analysis 5: Modularity Over Time

In [ ]:
# Compute Newman modularity per window using GICS sectors as communities
def compute_modularity_over_time(adjacencies, tickers, sectors):
    """Compute modularity with GICS sectors as communities per window."""
    unique_sectors = sorted(set(sectors))
    sector_to_idx = {s: i for i, s in enumerate(unique_sectors)}
    community_labels = [sector_to_idx[s] for s in sectors]
    
    modularities = []
    for t in range(len(adjacencies)):
        adj = adjacencies[t].copy()
        np.fill_diagonal(adj, 0)
        G = nx.from_numpy_array(adj)
        
        # Create partition as list of sets
        communities = [set() for _ in range(len(unique_sectors))]
        for i, label in enumerate(community_labels):
            communities[label].add(i)
        communities = [c for c in communities if len(c) > 0]
        
        mod = nx.community.modularity(G, communities)
        modularities.append(mod)
        
        if t % 200 == 0:
            print(f"  Processed {t+1}/{len(adjacencies)} windows")
    
    return np.array(modularities)

print("Computing modularity over time...")
modularities = compute_modularity_over_time(gcn_adj, available_tickers, sectors)
print(f"Done. Mean modularity: {modularities.mean():.4f}")

In [ ]:
# Plot modularity over time with rolling Sharpe overlay
from gml.connectivity_analysis import plot_connectivity_vs_performance

# Compute rolling Sharpe (matching 5b)
window_returns = gcn_daily.reindex(pd.to_datetime(gcn_dates)).values
rolling_sharpe = pd.Series(window_returns).rolling(20).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0
).values

fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.plot(pd.to_datetime(gcn_dates), modularities, color="#1f77b4", lw=0.8, label="Modularity")
ax1.set_ylabel("Modularity (GICS sectors)", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")

ax2 = ax1.twinx()
ax2.plot(pd.to_datetime(gcn_dates), rolling_sharpe, color="#d62728", lw=0.8, alpha=0.7, label="Rolling Sharpe")
ax2.set_ylabel("Rolling Sharpe (20-window)", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.set_title("Graph Modularity (sector alignment) vs Model Performance")
ax1.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

# Correlation
valid_mask = ~np.isnan(rolling_sharpe)
r, p = stats.pearsonr(modularities[valid_mask], rolling_sharpe[valid_mask])
print(f"Correlation (modularity vs rolling Sharpe): r={r:.4f}, p={p:.4f}")
print(f"{'SIGNIFICANT' if p < 0.05 else 'NOT significant'}")

## 10. Analysis 6: Degree Heatmap Over Time

In [ ]:
# Heatmap: per-stock degree over time
fig, ax = plt.subplots(figsize=(16, 10))

# Sort tickers by sector for visual grouping
sector_order = sorted(range(len(available_tickers)), key=lambda i: (sectors[i], available_tickers[i]))
sorted_tickers = [available_tickers[i] for i in sector_order]
sorted_degrees = node_features["degree"][:, sector_order]

# Subsample time for visibility (every 10th window)
step = 10
sns.heatmap(sorted_degrees[::step].T, ax=ax, cmap="YlOrRd",
            yticklabels=sorted_tickers, xticklabels=False)
ax.set_xlabel("Time (windows)")
ax.set_ylabel("Stock (grouped by sector)")
ax.set_title("Per-Stock Degree Over Time (sorted by GICS sector)")
plt.tight_layout(); plt.show()

## 11. Summary

In [ ]:
print("=" * 60)
print("5bii: NODE-LEVEL GRAPH FEATURES SUMMARY")
print("=" * 60)

print(f"
1. Degree vs Prediction Quality:")
print(f"   Mean cross-sectional correlation: {corr_df['r'].mean():.4f}")
t_stat, p_val = stats.ttest_1samp(corr_df["r"].dropna(), 0)
print(f"   Significant? {'YES' if p_val < 0.05 else 'NO'} (p={p_val:.4f})")

print(f"
2. Hub Stocks:")
print(f"   {len(hub_tickers)} hub stocks identified (top 25% by degree)")
print(f"   Hub Sharpe: {sharpe_ratio(hub_returns):.3f} vs Non-Hub: {sharpe_ratio(non_hub_returns):.3f}")

print(f"
3. Clustering Coefficient:")
print(f"   Mean cross-sectional correlation: {cluster_corr_df['r'].mean():.4f}")

print(f"
4. Eigenvector Centrality:")
print(f"   Mean cross-sectional correlation: {cent_corr_df['r'].mean():.4f}")

print(f"
5. Modularity:")
print(f"   Mean modularity: {modularities.mean():.4f}")
r_mod, p_mod = stats.pearsonr(modularities[valid_mask], rolling_sharpe[valid_mask])
print(f"   Correlation with Sharpe: r={r_mod:.4f}, p={p_mod:.4f}")